# DDPM: от шума к выборке

**Denoising Diffusion Probabilistic Models** · Ho, Jain & Abbeel · 2020

[Оригинальная статья](https://arxiv.org/abs/2006.11239)

**Цель урока:** Написать forward noising, epsilon predictor и полный маленький reverse chain на одномерных данных.

## Перед началом

Gaussian noise, MSE, оптимизатор и вводный world-model урок.

Около 40–60 минут с заданиями. CPU, PyTorch и Matplotlib; без скачивания весов и датасетов. Время — ориентир для чтения и работы, а не benchmark выполнения.

## Задача и идея

Diffusion постепенно разрушает данные шумом и учится обращать этот процесс. Модель получает x_t и t, предсказывая добавленный noise. На inference настоящего x0 уже нет: генерация начинается с Gaussian noise.

Механизм полезен для понимания diffusion world models, но сам по себе не моделирует действие и время среды. Здесь каждый объект — одно число из смеси двух Gaussians. Это осознанно маленькая постановка для проверки формул, не генератор изображений или DIAMOND.

## Граница воспроизведения

Одномерный DDPM с tiny MLP и коротким schedule. Нет U-Net, image benchmarks, classifier-free guidance или world-model rollout.

Код написан для этого урока по математическим идеям статьи. Это не скопированный авторский репозиторий. Успех assertions подтверждает указанные свойства реализации, но не результаты статьи.

## Формула → тензоры

`alpha_t = 1-beta_t`; `abar_t = product(alpha_0...alpha_t)`. `x_t = sqrt(abar_t)*x0 + sqrt(1-abar_t)*epsilon`. Loss=`MSE(epsilon_theta(x_t,t), epsilon)`.

Reverse mean: `(x_t - beta_t/sqrt(1-abar_t)*eps_hat)/sqrt(alpha_t)`. Здесь используем fixed posterior variance `beta_t*(1-abar_prev)/(1-abar_t)`. На последнем шаге t=0 шум не добавляем. Время diffusion и время rollout среды — разные оси.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Шаг 1

Проверяем schedule и точное восстановление x0, если известен настоящий noise. Это oracle sanity check, не генерация.

In [ ]:
T = 80
beta = torch.linspace(1e-4, 0.2, T)
alpha = 1 - beta; abar = alpha.cumprod(0)
previous = torch.cat([torch.ones(1), abar[:-1]])
posterior_variance = beta * (1 - previous) / (1 - abar)
assert (abar[1:] < abar[:-1]).all() and posterior_variance[0] == 0
x0 = torch.randn(128, 1); epsilon = torch.randn_like(x0); t = torch.randint(T, (128,))
xt = abar[t, None].sqrt() * x0 + (1 - abar[t, None]).sqrt() * epsilon
recovered = (xt - (1 - abar[t, None]).sqrt() * epsilon) / abar[t, None].sqrt()
assert torch.allclose(recovered, x0, atol=1e-4)
print('terminal alpha_bar:', abar[-1].item())

## Шаг 2

Учим маленькую MLP предсказывать epsilon. Данные генерируются заново на каждом шаге.

In [ ]:
def data(batch):
    return (torch.randint(0, 2, (batch, 1)).float() * 2 - 1) * 2 + 0.3 * torch.randn(batch, 1)
predictor = nn.Sequential(nn.Linear(2, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 1))
def predict(x, time):
    return predictor(torch.cat([x, time[:, None].float() / (T - 1)], -1))
optimizer = torch.optim.Adam(predictor.parameters(), lr=0.003)
history = []
for step in range(400):
    clean = data(128); time = torch.randint(T, (128,)); noise = torch.randn_like(clean)
    noisy = abar[time, None].sqrt() * clean + (1 - abar[time, None]).sqrt() * noise
    loss = F.mse_loss(predict(noisy, time), noise)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step(); history.append(loss.item())
assert math.isfinite(history[-1])
plt.plot(history, color='#d73535', alpha=0.6); plt.xlabel('step'); plt.ylabel('epsilon MSE'); plt.show()

## Шаг 3

Reverse chain не видит clean data. Сравнение гистограмм диагностическое: маленькая сеть может давать плохие выборки.

In [ ]:
with torch.no_grad():
    samples = torch.randn(1000, 1)
    for index in reversed(range(T)):
        time = torch.full((len(samples),), index, dtype=torch.long)
        epsilon_hat = predict(samples, time)
        mean = (samples - beta[index] / (1 - abar[index]).sqrt() * epsilon_hat) / alpha[index].sqrt()
        samples = mean + posterior_variance[index].sqrt() * torch.randn_like(samples) if index else mean
assert torch.isfinite(samples).all()
plt.hist(data(1000).flatten(), bins=50, density=True, alpha=0.4, color='gray', label='data')
plt.hist(samples.flatten(), bins=50, density=True, alpha=0.5, color='#d73535', label='generated')
plt.legend(); plt.xlabel('scalar value'); plt.show()

## Как читать результат

Train loss усредняет разные noise levels; отдельно полезно смотреть MSE по t. Ошибка индекса schedule или дополнительный noise на t=0 может испортить sampling при рабочем backward. Совпадение средних не доказывает совпадение распределений.

## Самостоятельные эксперименты

1. Уберите t из predictor.
2. Сделайте terminal abar близким к 1: корректен ли Gaussian старт?
3. Добавьте conditioning на действие и объясните, откуда взять transition dataset.

Сначала запишите прогноз, затем меняйте по одному фактору. Для честного сравнения сбрасывайте ядро и выполняйте все ячейки с начала. Сохраните копию для своих изменений.

## Проверка рассуждений — после эксперимента

Без t сеть смешивает разные задачи denoising. Если terminal data остаётся заметной, standard normal старт хуже соответствует forward endpoint. Для action-conditioned world model нужны observation/action/next-observation triples и независимые эпизоды для оценки.

## Что записать в лабораторный журнал

- Гипотеза и изменённый параметр.
- Seed, размер данных, число шагов и используемая precision.
- Значения проверок до и после, график и объяснение расхождения с прогнозом.
- Что этот эксперимент **не** позволяет заключить.

Связанный раздел доступен по кнопке «Открыть раздел студии» над уроком.